# LiSSA on Colab (L4): RQ1 to RQ3

**Every session, including after a disconnect:** use an **L4 GPU** runtime, run cells 1 to 4 in order, then run the cell of the step you are on again. It continues where it stopped.

**What is saved, and where:** Google Drive `MyDrive/lissa-colab/`. The answer caches, results, harness runs and prompts are copied there every 2 minutes. Each file is written to a temporary name first and renamed when complete, so an interrupted copy never replaces a good one. An hourly snapshot goes to `MyDrive/lissa-colab-snapshots/`, keeping the last 6.

**Worst case after a crash:** the last up-to-50 answers LiSSA had not yet written, plus at most 2 minutes. The run re-asks only those.

**Cell 1: connect Google Drive and check the GPU** (it stops if the GPU is not an L4, so every run uses the same GPU type)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)
assert "L4" in gpu, "Not an L4. Change the runtime type to L4, then start again."

Mounted at /content/drive
NVIDIA L4, 23034 MiB


**Cell 2: install everything (about 3 to 5 minutes)**

In [ ]:
%%bash
cd /content
[ -d LiSSA-Replication-Package ] || git clone -q --depth 1 -b rq2-rq3-harness https://github.com/rahulxdagar/LiSSA-Replication-Package.git
[ -d Replication-Package-ICSE25_LiSSA-Toward-Generic-Traceability-Link-Recovery-through-RAG ] || git clone -q --depth 1 https://github.com/ardoco/Replication-Package-ICSE25_LiSSA-Toward-Generic-Traceability-Link-Recovery-through-RAG.git
rm -f LiSSA-Replication-Package/harness/data/candidates-local.json

apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y openjdk-21-jdk-headless maven zstd rsync > /dev/null 2>&1
printf '#!/bin/sh\necho /usr/lib/jvm/java-21-openjdk-amd64\n' > /usr/libexec/java_home && chmod +x /usr/libexec/java_home

curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.2 sh > /dev/null
pip -q install -r LiSSA-Replication-Package/harness/requirements.txt
/usr/lib/jvm/java-21-openjdk-amd64/bin/java -version 2>&1 | grep -i version; mvn -v 2>/dev/null | grep -i "apache maven"; ollama -v 2>&1 | tail -1

openjdk version "21.0.13.0.3" 2026-10-24
Apache Maven 3.8.7


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


**Cell 3: save to and restore from Google Drive** (it refuses to run when Drive is not connected)

In [ ]:
%%bash
cat > /content/sync.sh <<'EOF'
REPO=/content/LiSSA-Replication-Package
SAVE=/content/drive/MyDrive/lissa-colab
SNAP=/content/drive/MyDrive/lissa-colab-snapshots
PATHS="LiSSA-RATLR-V2/lissa/cache-local-r2c LiSSA-RATLR-V2/lissa/results-local harness/cache harness/runs harness/data harness/prompts"
if ! mountpoint -q /content/drive; then echo "$(date -u +%FT%TZ) Google Drive is not connected: nothing saved or restored. Run cell 1."; exit 1; fi
case "$1" in
  save)
    for p in $PATHS; do [ -d "$REPO/$p" ] && mkdir -p "$SAVE/$p" && rsync -a "$REPO/$p/" "$SAVE/$p/"; done
    date -u +%FT%TZ > "$SAVE/last-save.txt" ;;
  restore)
    for p in $PATHS; do [ -d "$SAVE/$p" ] && mkdir -p "$REPO/$p" && rsync -a "$SAVE/$p/" "$REPO/$p/"; done
    echo "restored from Drive (last save: $(cat "$SAVE/last-save.txt" 2>/dev/null || echo never))" ;;
  snapshot)
    mkdir -p "$SNAP"; cd "$REPO" && tar czf "$SNAP/.tmp.tgz" $(for p in $PATHS; do [ -d "$p" ] && echo "$p"; done) \
      && mv "$SNAP/.tmp.tgz" "$SNAP/snapshot-$(date -u +%Y%m%d-%H%M).tgz"
    ls -1t "$SNAP"/snapshot-*.tgz 2>/dev/null | tail -n +7 | xargs -r rm -f ;;
esac
EOF
bash /content/sync.sh restore

restored from Drive (last save: never)


**Cell 4: start Ollama, save every 2 minutes, snapshot every hour**

In [ ]:
import os, subprocess, time, urllib.request
os.chdir("/content/LiSSA-Replication-Package")
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_CONTEXT_LENGTH": "16384"}
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
autosave = subprocess.Popen(["bash", "-c",
    "i=0; while true; do sleep 120; bash /content/sync.sh save >> /content/sync.log 2>&1; "
    "i=$((i+1)); [ $((i % 30)) -eq 0 ] && bash /content/sync.sh snapshot >> /content/sync.log 2>&1; done"])
RQ1, CACHE = "LiSSA-RATLR-V2/lissa/results-local/rq1", "LiSSA-RATLR-V2/lissa/cache-local-r2c"
time.sleep(5)
print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())

{"version":"0.40.2"}


**RQ1, full run** (one seed; about 16 hours on an L4). After a disconnect, run cells 1 to 4, then this cell again: it continues from the saved answers.

In [6]:
!python3 run_rq1_benchmark.py --seeds 1 --hours 20
!bash /content/sync.sh save && bash /content/sync.sh snapshot && cat /content/drive/MyDrive/lissa-colab/last-save.txt

Streaming output truncated to the last 5000 lines.
2026-11-15 12:38:54:299 +0000 [pool-1-thread-1] INFO SimpleClassifier - Classifying: SMOS14.txt and UserTest.txt
2026-11-15 12:38:54:552 +0000 [pool-1-thread-1] INFO SimpleClassifier - Classifying: SMOS14.txt and ServletShowClassroomByAcademicYear.txt
2026-11-15 12:38:54:872 +0000 [pool-1-thread-1] INFO SimpleClassifier - Classifying: SMOS14.txt and LoginException.txt
2026-11-15 12:38:55:088 +0000 [pool-1-thread-1] INFO SimpleClassifier - Classifying: SMOS14.txt and ServletDeleteClassroom.txt
2026-11-15 12:38:55:418 +0000 [pool-1-thread-1] INFO SimpleClassifier - Classifying: SMOS14.txt and ServletAddTeachingAsTeacher.txt
2026-11-15 12:38:56:026 +0000 [pool-1-thread-1] INFO SimpleClassifier - Classifying: SMOS14.txt and User.txt
2026-11-15 12:38:56:530 +0000 [pool-1-thread-1] INFO SimpleClassifier - Classifying: SMOS14.txt and ServletShowTeacherTeachingFormByClass.txt
2026-11-15 12:38:57:024 +0000 [pool-1-thread-1] INFO SimpleClassifie

**RQ2 and RQ3:** follow `harness/RQ2_RQ3_INSTRUCTIONS.md` (section A4) with `!` in front of each command, and add `--device colab-l4` to every `run.py` and `verify.py` command.
The first three steps:

In [ ]:
!python3 harness/check_rq1.py --rq1-run $RQ1 --lissa-cache $CACHE --reask 20
!python3 harness/prepare_prompts.py --force-examples
!python3 harness/selftest.py
!bash /content/sync.sh save

**Before stopping:** save, then use Runtime > Disconnect and delete runtime, so compute units stop being used.

In [ ]:
!bash /content/sync.sh save && bash /content/sync.sh snapshot && cat /content/drive/MyDrive/lissa-colab/last-save.txt